## Imports

In [ ]:
import yfinance as yf
import pandas as pd
import os
import boto3
from datetime import *
from dotenv import load_dotenv
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import pyarrow

## Conection AWS S3

In [ ]:
load_dotenv()

ACCESS_KEY = os.getenv('ACCESS_KEY')
SECRET_KEY = os.getenv('SECRET_KEY')

client = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

# Bronze Layer

In [ ]:
df = yf.download(['AAPL', 'NVDA'], period='max', interval='1d', multi_level_index=False, actions=True)

In [ ]:
today = datetime.now().strftime('%d-%m-%y')
os.makedirs('input', exist_ok=True)
df.to_csv(f'input/stocks_{today}.csv', index=False)

In [ ]:
# Broze layer of data saved in S3 bucket.
client.upload_file(Filename=f'input/stocks_{today}.csv', Bucket='ace-low', Key=f'finance/stocks_{today}.csv')

# Silver Layer

In [ ]:
df = df.stack(level='Ticker').reset_index()
df.columns.name = None
df.head(8)

## EDA and Data cleaning

In [ ]:
df.columns = (df.columns.str.upper().str.replace(' ', '_'))
df = df[['DATE', 'OPEN', 'LOW', 'HIGH', 'CLOSE', 'STOCK_SPLITS', 'DIVIDENDS']]

In [ ]:
df.info()

In [ ]:
# sorting values by highest date
df = df.sort_values(by='DATE', ascending=False)

df['VARIATION_PAST_DAY'] = df['CLOSE'].diff()

df['PERCENT_VARIATION_PAST_DAY'] = df['CLOSE'].pct_change()

df['GAP_PAST_DAY'] = (df['OPEN'] - df['CLOSE']).shift(1)

df['FLAG_DAY'] = np.where(
    df['OPEN'] < df['CLOSE'],
    'POSITIVE',
    np.where(
        df['OPEN'] == df['CLOSE'],
        'EQUAL',
        'NEGATIVE'
    )
)

df['INSERT_DATE'] = datetime.now()
df['INSERT_DATE'] = df['INSERT_DATE'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['DATE'] = df['DATE'].dt.strftime('%Y-%m-%d %H:%M:%S')

## Connect Snowflake

In [ ]:
## Get env keys
load_dotenv()
USER_SNOW = os.getenv('USER_SNOW')
PASSWORD_SNOW = os.getenv('PASSWORD_SNOW')
ACCOUNT_SNOW = os.getenv('ACCOUNT_SNOW')

con_snow = snowflake.connector.connect(
    user=USER_SNOW,
    password=PASSWORD_SNOW,
    account=ACCOUNT_SNOW
)

### Create DW and table

In [ ]:
con_snow.cursor().execute("CREATE WAREHOUSE IF NOT EXISTS ace")
con_snow.cursor().execute("CREATE DATABASE IF NOT EXISTS ace_low")
con_snow.cursor().execute("USE DATABASE ace_low")
con_snow.cursor().execute("CREATE SCHEMA IF NOT EXISTS stocks")

con_snow.cursor().execute("""
                          CREATE TABLE IF NOT EXISTS stocks.stock_market(
                            id integer PRIMARY KEY AUTOINCREMENT,
                            date DATETIME NOT NULL,
                            open float,
                            low float,
                            high float,
                            close float,
                            stock_splits float,
                            dividends float,
                            variation_past_day float,
                            percent_variation_past_day float,
                            gap_past_day float,
                            flag_day string,
                            insert_date datetime
                          )
                        """)

In [ ]:
success, nchunks, nrows, _ = write_pandas(
    conn=con_snow,
    df=df,
    schema='STOCKS',
    table_name='STOCK_MARKET'
)